# DT401 Week 2 Section 3: Data Formats — CSV, JSON, and Markdown

**Module:** Data Fundamentals  
**Week:** 2 of 7  
**Section:** 3 of 4  
**Estimated Time:** 25–30 minutes

---

## Learning Objectives

By the end of this notebook, you will be able to:

- Recognise the structure of CSV, JSON, and Markdown formats by reading real examples
- Explain when each format is appropriate based on data characteristics and use case
- Identify how data formats connect to the data structures (tables, key-value pairs) explored in Section 2
- Relate data formats to SQL concepts you will encounter in Section 4 and Workshop 1

---

## Context Recap

In Section 2, you explored how data is organised into structures — tables, lists, dictionaries, and trees — and saw how those structures appear in Python and SQL. This notebook adds the next layer: **formats**. A format is how a structure is written down and shared. The same data can be stored as a CSV file, transmitted as JSON, or displayed in a SQL table — the information stays the same; only the representation changes.

Section 4 introduces SQL, where you will write your own queries. What you observe in this notebook — particularly how CSV and SQL tables represent the same tabular data — is the foundation you will build on there.

---

## A Note Before You Start

This notebook is about **exploring** data formats — not writing code. Every code cell in this notebook has already been written for you. Your job is to **run** each cell, **read** the output, and **think** about what it means.

You are not expected to write Python from scratch yet. That comes in DT402. Here, the code is a window into the data — you are looking through it, not building it.

One task in Part 1 will ask you to change a single value in a code cell and re-run it to see what happens. Two tasks will ask you to fill in a table directly in a markdown cell. Instructions will always be explicit about exactly what to do.

Section 4 is where you will start writing code yourself — specifically, SQL queries to create and retrieve data from database tables.

---

## How to Use This Notebook

You will find three types of box throughout this notebook:

<div class="alert alert-block alert-warning">
<b>Reminder boxes:</b><br>
Important context about how this notebook connects to the other material you are studying or to your Module Challenge.
</div>

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>
Explanations of what a code cell did and what you should take from the output.
</div>

<div class="alert alert-block alert-success">
<b>Your turn:</b><br>
Tasks for you to complete. Some involve editing a single value in a code cell. Others ask you to fill in a table in a markdown cell. Instructions will always be explicit — you will never be asked to write code from scratch.

Tasks may include a hidden solution.
<details>
    <summary style="color:black;font-weight:bold">Click here to reveal the solution</summary>

Solutions appear here.

</details>
</div>

---

## Setup

Run the cell below to load the libraries this notebook uses and connect to a temporary database. You should see a confirmation message.

In [ ]:
import pandas as pd
import json
from io import StringIO
from IPython.display import display
%config SqlMagic.displaylimit = 0
%config SqlMagic.autopandas = True
%load_ext sql
%sql sqlite:///:memory:

# Load the same weather dataset used throughout Week 2
df = pd.read_csv('../open-meteo-51.49N0.16W23m.csv', skiprows=3, nrows=26328, low_memory=False,
                 converters={'is_day ()': lambda x: bool(float(x))})

# Work with a focused set of columns consistent with Section 2
weather_cols = ['time', 'temperature_2m (°C)', 'rain (mm)', 'is_day ()']

print("Setup complete — pandas, json, and database ready.")
print(f"Weather dataset loaded: {len(df)} observations, {len(df.columns)} columns")

---

## Part 1: CSV — Tabular Data as Text

### What is CSV?

You have seen tables throughout this module — DataFrames in Python, spreadsheets in your workplace, the weather dataset displayed row by row. **CSV (Comma-Separated Values)** is how those tables look when saved as a plain text file. There is no special software involved: commas separate columns, and newlines separate rows. That is the entire format.

CSV is the universal export format for data. Customer lists, sales reports, inventory data, financial exports — every system can read and write CSV because the rules are so simple. When you download a report from a CRM, receive a data extract from a supplier, or export records from an NHS clinical system to share with a regional analytics team, the file almost always arrives as CSV.

<div class="alert alert-block alert-warning">
<b>Workplace connection:</b><br>
Think about data exports from systems you use at work — reports downloaded from a CRM, patient data exported from a clinical system, financial data from an accounting package. Many of these arrive as CSV files. You are about to look at what is inside them.
</div>

### Worked Example 1: Viewing raw CSV text

**What you will see:** The first four lines of the weather CSV file printed as raw text, followed by the same data loaded as a pandas DataFrame. The connection between the two representations is the point.

In [ ]:
# Read the first 4 lines of the CSV file as raw text — no parsing, just the characters on disk
with open('../open-meteo-51.49N0.16W23m.csv', 'r') as f:
    # Skip the 3 metadata rows at the top of this particular file
    for _ in range(3):
        f.readline()
    raw_lines = [f.readline().strip() for _ in range(4)]

print("Raw CSV text (first 4 lines):")
print("---")
for line in raw_lines:
    # Truncate long lines for readability — the full line has 33 columns
    print(line[:120] + ("..." if len(line) > 120 else ""))

print("\n--- The same data loaded as a pandas DataFrame ---\n")
display(df[weather_cols].head(3))

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

The raw text and the DataFrame contain exactly the same data. CSV is just text with one rule applied twice: **commas separate columns; newlines separate rows**. The first line is the header row — it gives each column its name. Every line after that is one record.

pandas reads this text and builds a table from it. The column names come from that first header line. The DataFrame you see below is not a different kind of data — it is the same CSV text, displayed in a more readable form.
</div>

### Worked Example 2: A workplace CSV example

**What you will see:** A small CSV string representing NHS clinic appointment data, first as raw text and then as a pandas DataFrame.

In [ ]:
# NHS clinic appointment data as a CSV string
# This is exactly what the file looks like if you open it in a text editor
clinic_csv = """date,patient_id,appointment_type,duration_mins,attended
2026-01-06,PAT-0471,GP Consultation,15,True
2026-01-06,PAT-0512,Blood Test,10,True
2026-01-07,PAT-0388,Follow-up,20,False"""

print("Raw CSV text:")
print("---")
print(clinic_csv)

print("\n--- The same data as a pandas DataFrame ---\n")
# StringIO lets pandas read a string as if it were a file
clinic_df = pd.read_csv(StringIO(clinic_csv))
display(clinic_df)

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

CSV stores one record per line, one field per comma. Notice the column types that pandas inferred: `date` and `patient_id` are strings, `duration_mins` is an integer, and `attended` is a boolean. The structure is flat — each row is independent, and every row has the same columns.

This is the same table structure from Section 2 — but now stored in a format that any system can read without specialist software. This is why CSV is the default format for sharing data between organisations.
</div>

<div class="alert alert-block alert-info">
<b>Reinforcement Resource</b><br>

[CSV files — European Data Portal (European Data Portal, n.d.)](https://data.europa.eu/apps/data-visualisation-guide/csv-files) — A concise guide to CSV as a data exchange format: its structure, near-universal compatibility, and common pitfalls to watch for.

*Read time: 5 minutes*
</div>

### SQL Preview: The Same Data in Three Forms

**What you will see:** The same three clinic appointment records represented as raw CSV text (above), as a pandas DataFrame (above), and now as a SQL table.

In [ ]:
%%sql
CREATE TABLE IF NOT EXISTS clinic_appointments (
    date             TEXT,
    patient_id       TEXT,
    appointment_type TEXT,
    duration_mins    INTEGER,
    attended         BOOLEAN
);

INSERT INTO clinic_appointments VALUES
    ('2026-01-06', 'PAT-0471', 'GP Consultation', 15, TRUE),
    ('2026-01-06', 'PAT-0512', 'Blood Test',      10, TRUE),
    ('2026-01-07', 'PAT-0388', 'Follow-up',       20, FALSE);

SELECT * FROM clinic_appointments;

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

The same three records appear in all three representations. **CSV** is how the data travels between systems — any application can open it. A **DataFrame** is how Python pandas works with it in memory. A **SQL table** is how a database stores and queries it.

In Section 4, you will write SQL like this yourself. Notice how the column names and data types in the `CREATE TABLE` statement map directly to the columns in the CSV header row — this is not a coincidence. They describe the same structure.
</div>

<div class="alert alert-block alert-success">
<b>Your turn — guided modification:</b><br>

In Worked Example 2 above, the clinic data shows three appointments. Find the row for `PAT-0471` and change the `duration_mins` value from `15` to `30`.

Re-run that cell. Observe that both the raw CSV text and the DataFrame update to show the new value.

**The point:** The CSV text and the DataFrame are two views of the same data. Change the source text, and the table changes too.
</div>

---

## Part 2: JSON — Nested Data as Text

### What is JSON?

In Section 2, you worked with Python dictionaries — key-value pairs that store structured records. **JSON (JavaScript Object Notation)** is how those dictionaries look when sent between systems. It is the standard format for web APIs, configuration files, and modern data exchange.

When your organisation's website retrieves product information from a catalogue service, when a weather provider sends forecast data to a dashboard, when a mobile banking app syncs transaction history — JSON is the format being transmitted. NHS Digital uses JSON for its FHIR APIs (patient summary data); Monzo structures its transaction records as JSON objects. Understanding JSON means you can read and discuss what those systems are exchanging.

```mermaid
flowchart TD
    A[JSON Document] --> B[Objects — curly braces]
    A --> C[Arrays — square brackets]
    A --> D[Key-Value Pairs]
    B --> E["{'time': '2023-01-01T00:00'}"]
    C --> F["[11.2, 11.9, 11.0]"]
    D --> G["Key names describe the data"]

    style A fill:#ffffff,stroke:#000000,color:#000000,stroke-width:2px
    style B fill:#ffffff,stroke:#000000,color:#000000,stroke-width:2px
    style C fill:#ffffff,stroke:#000000,color:#000000,stroke-width:2px
    style D fill:#ffffff,stroke:#000000,color:#000000,stroke-width:2px
    style E fill:#ffffff,stroke:#000000,color:#000000,stroke-width:1px
    style F fill:#ffffff,stroke:#000000,color:#000000,stroke-width:1px
    style G fill:#ffffff,stroke:#000000,color:#000000,stroke-width:1px
```

**What this diagram shows:** JSON uses three building blocks — objects (curly braces, like Python dictionaries), arrays (square brackets, like Python lists), and key-value pairs. These are the same structures from Section 2, written in a standardised text format that any programming language can read.

### Worked Example 1: One Weather Observation as JSON

**What you will see:** One row from the weather DataFrame converted first to a Python dictionary, then to a JSON string. The two representations are nearly identical.

In [ ]:
# Extract the first observation and convert to a dictionary
observation_dict = df[weather_cols].iloc[0].to_dict()

print("As a Python dictionary:")
display(observation_dict)

print("\nAs a JSON string (ready to send over a network or save to a file):")
# json.dumps() converts a Python dictionary to a JSON-formatted string
# indent=2 adds indentation so the structure is easy to read
print(json.dumps(observation_dict, indent=2))

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

A Python dictionary and a JSON object are almost the same thing. `json.dumps()` converts one to the other. The key-value structure you explored in Section 2 — field names mapped to field values — is exactly the structure JSON uses for data interchange.

This is the same observation you saw as a DataFrame row in Section 2. Now it is a format that can travel across a network or sit inside a configuration file.
</div>

### Worked Example 2: Nested JSON — A Richer Record

**What you will see:** A nested JSON record representing an NHS patient referral. JSON can hold objects within objects and lists within objects — this is the tree structure from Section 2.

In [ ]:
# An NHS patient referral — a nested record
# Notice that gp_practice is itself a dictionary, and appointment_dates is a list
patient_referral = {
    "referral_id": "REF-2026-0847",
    "patient": {
        "name": "Priya Sharma",
        "nhs_number": "485 777 3456",
        "date_of_birth": "1991-03-14"
    },
    "gp_practice": {
        "practice_name": "Elmfield Road Surgery",
        "gp_name": "Dr Kwame Mensah",
        "ods_code": "E84012"
    },
    "appointment_dates": [
        "2026-02-10",
        "2026-03-04"
    ],
    "urgent": False
}

print("NHS patient referral as JSON:")
print(json.dumps(patient_referral, indent=2))

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

JSON supports nesting — objects within objects, arrays within objects. This is the tree structure from Section 2. The patient referral contains a `patient` sub-record and a `gp_practice` sub-record, each with their own fields. The `appointment_dates` field is a JSON array — a list of dates.

CSV cannot represent this nesting. A CSV row is flat — every field sits at the same level. JSON can represent hierarchical data like this naturally, which is why APIs use JSON rather than CSV for complex records.
</div>

### Worked Example 3: Comparing the Same Data in CSV and JSON

**What you will see:** Three weather observations printed first as CSV text, then as JSON. Same data; different representations.

In [ ]:
# Select 3 weather observations with a focused set of columns
comparison_cols = ['time', 'temperature_2m (°C)', 'rain (mm)']
sample = df[comparison_cols].head(3)

print("CSV format (compact, flat):")
print("---")
print(sample.to_csv(index=False))

print("JSON format (self-describing, structured):")
print("---")
print(sample.to_json(orient='records', indent=2))

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

The same three observations in two formats. CSV is shorter — the column names appear once at the top, and every row reuses that structure implicitly. JSON is more verbose — every value carries its key name, so each record is self-describing. Either format can be opened by the right tool, but they are designed for different purposes.

CSV is good for flat tables that will be opened in a spreadsheet or loaded into a database. JSON is good for APIs, where the receiver needs to know what each value represents without a separate header row, and where records may have nested or variable structure.
</div>

<div class="alert alert-block alert-info">
<b>Reinforcement Resource</b><br>

[JSON Introduction — W3Schools (W3Schools, 2026)](https://www.w3schools.com/js/js_json_intro.asp) — A concise overview of JSON syntax and structure, showing how key-value pairs are formatted for data interchange.

*Read time: 5 minutes*
</div>

<div class="alert alert-block alert-success">
<b>Your turn — classification task:</b><br>

For each scenario below, decide whether **CSV** or **JSON** is the more appropriate format. Edit this cell to fill in the empty columns — write the format name and a brief reason (one sentence is enough).
</div>

| Scenario | CSV or JSON? | Reason |
|:---------|:------------|:-------|
| Exporting 10,000 rows of sales transactions to open in Excel | | |
| Sending a customer profile with nested address and order history to a mobile app | | |
| Sharing a simple list of product IDs and prices with a supplier | | |
| Receiving real-time weather data from a web API | | |
| Archiving historical employee records for compliance purposes | | |

<details>
    <summary style="font-weight:bold">Click here to reveal the solution</summary>

| Scenario | CSV or JSON? | Reason |
|:---------|:------------|:-------|
| Exporting 10,000 rows of sales transactions to open in Excel | CSV | Flat tabular data; Excel opens CSV natively; no nesting required |
| Sending a customer profile with nested address and order history to a mobile app | JSON | The record has nested sub-objects (address, order history); JSON represents this naturally; it is the standard format for API responses |
| Sharing a simple list of product IDs and prices with a supplier | CSV | Two columns, flat structure; CSV is compact and universally readable |
| Receiving real-time weather data from a web API | JSON | Web APIs standardise on JSON; the data may include nested or variable fields depending on the provider |
| Archiving historical employee records for compliance purposes | CSV | Flat records; CSV is durable, human-readable, and does not depend on any particular software to open — important for long-term archiving |

</details>

---

## Part 3: Markdown and Format Selection

### What is Markdown?

You have already been reading **Markdown** throughout this module — every text cell in this notebook is written in it. Markdown is a lightweight formatting language: you add simple symbols to plain text (a `#` before a heading, `**` around bold words, `-` before a list item) and those symbols control how the text appears when rendered.

Markdown is different from CSV and JSON in one important way: it is designed primarily for **human reading**, not machine processing. Its purpose is documentation. README files in software repositories, notes in Jupyter notebooks, project documentation on platforms like Confluence, and messages formatted in Slack or Teams all use Markdown. You have been using it without necessarily knowing the syntax.

### Worked Example: Markdown in This Notebook

The cell below shows some of the Markdown syntax you have been reading throughout Week 2. Each line of raw syntax is followed by an explanation of what it produces.

**Markdown syntax you have been reading in this notebook:**

```markdown
# Heading 1          → Large heading (used for notebook titles)
## Heading 2         → Section heading (used for Part headers like "## Part 3")
### Heading 3        → Subsection heading

**bold text**        → Bold (used for key terms like **CSV** on first use)
*italic text*        → Italic (used for emphasis)

- Item one           → Bullet list
- Item two

1. First step        → Numbered list
2. Second step

[Link text](URL)     → Hyperlink (used in Reinforcement Resource cells)

```code```           → Inline code (used for `column_names` and function names)

| Column | Column |  → Table (used for classification tasks like the one in Part 2)
|:-------|:-------|
| Cell   | Cell   |
```

Every formatted text element you have read in this notebook was produced by one of these patterns. The heading `## Part 3: Markdown and Format Selection` above this section used `##`. The word **CSV** in bold used `**CSV**`. The classification table in Part 2 used the table syntax shown here.

Markdown's value is that it is readable even without rendering — the raw syntax is legible as plain text — while producing clean, structured output when displayed in a notebook, browser, or documentation tool.

As Markdown manages to be both human-readable and structured, it is ideal for Large Language Models (LLMs) like ChatGPT. When you ask an LLM to produce a table, it will often format it in Markdown. This is because Markdown tables are easy for the model to generate and for you to read — they strike a balance between structure and readability.

<div class="alert alert-block alert-info">
<b>Reinforcement Resource</b><br>

[Markdown Cheat Sheet — Markdown Guide (Markdown Guide, 2026)](https://www.markdownguide.org/cheat-sheet/) — A quick-reference overview of all core Markdown syntax elements — the same syntax you have been reading throughout this notebook.

*Read time: 5 minutes*
</div>

### Format Selection: Choosing the Right Format

CSV, JSON, Markdown, and SQL are not competitors — they solve different problems. In practice, your organisation uses all of them. The table below summarises the key differences.

| Feature | CSV | JSON | Markdown | SQL Table |
|:--------|:----|:-----|:---------|:----------|
| **Structure** | Flat (rows and columns) | Nested (objects and arrays) | Sections and paragraphs | Flat (rows and columns) |
| **Best for** | Data exchange, spreadsheets | Web APIs, configuration | Documentation, notes | Large datasets, queries |
| **Human-readable?** | Somewhat | Yes | Yes | Via query results |
| **Supports nesting?** | No | Yes | No | Via related tables |
| **File size** | Compact | Larger | Small | N/A (stored in database) |
| **Your workplace** | Exports and reports | API integrations | Documentation and notes | Databases and reporting |

<div class="alert alert-block alert-warning">
<b>Workplace connection:</b><br>
In practice, your organisation uses all of these formats together. A database stores customer records in SQL tables. A report is exported as CSV for a manager to open in Excel. A web dashboard retrieves the same data as JSON from an API. The project documentation is written in Markdown. Understanding all four means you can participate in conversations across the full data pipeline.
</div>

---

## Part 4: Connecting Formats to Structures and to SQL

### Bringing It Together

In Section 2, you mapped data structures to their Python and SQL equivalents: a table maps to a DataFrame, a key-value pair maps to a dictionary, and so on. This Part adds the **file format layer**. Formats are how structures are saved and shared outside of a running programme.

The table below shows the complete mapping — from the conceptual structure, through Python, through the file format, to the SQL equivalent you will use in Section 4.

| What you are describing | Structure (Section 2) | Python equivalent | File format (Section 3) | SQL equivalent (Section 4) |
|:------------------------|:---------------------|:------------------|:------------------------|:---------------------------|
| A flat table of records | Table | DataFrame | CSV | TABLE |
| A single record with named fields | Key-Value Pair | Dictionary | JSON object | Row |
| A nested or hierarchical record | Tree | Nested dictionary | Nested JSON | Related tables with foreign keys |
| An ordered collection of values | List / Array | List | JSON array | Column |
| Human-readable documentation | — | — | Markdown | — |

**What this table shows:** The concepts stay the same — only the representation changes. A table is a table whether it lives in a CSV file, a pandas DataFrame, or a SQL database. A key-value pair is a key-value pair whether it is a Python dictionary, a JSON object, or a row in a SQL table. In Section 4, you will write SQL to create and query these structures directly.

### Worked Example: One Dataset, Four Representations

**What you will see:** Three weather observations in four forms — raw CSV text, JSON, a pandas DataFrame, and a SQL table. The data does not change.

In [ ]:
# Three weather observations — same data, different representations
demo_cols = ['time', 'temperature_2m (°C)', 'rain (mm)']
three_obs = df[demo_cols].head(3)

print("1. As CSV text:")
print("---")
print(three_obs.to_csv(index=False))

print("2. As JSON:")
print("---")
print(three_obs.to_json(orient='records', indent=2))

print("3. As a pandas DataFrame:")
print("---")
display(three_obs)

In [ ]:
%%sql
CREATE TABLE IF NOT EXISTS weather_formats_demo (
    time        TEXT,
    temperature DECIMAL(4,1),
    rain        DECIMAL(4,1)
);

INSERT INTO weather_formats_demo VALUES
    ('2023-01-01T00:00', 11.2, 0.1),
    ('2023-01-01T01:00', 11.9, 0.4),
    ('2023-01-01T02:00', 11.0, 0.0);

-- 4. As a SQL table:
SELECT * FROM weather_formats_demo;

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

You have now seen the same three weather observations as CSV text, JSON text, a pandas DataFrame, and a SQL table. This is the core insight of data formats: **the information is the same; the representation varies depending on the tool and the purpose**.

When you write SQL in Section 4, you will be working with the same structures you have explored throughout Sections 2 and 3. The column names, data types, and row-based organisation are all consistent — only the syntax changes.
</div>

<div class="alert alert-block alert-success">
<b>Your turn — comprehensive classification task:</b><br>

For each real-world data scenario below, identify (1) the most appropriate file format and (2) the underlying data structure from Section 2. Add a brief justification for your format choice — one or two sentences.

Edit this cell directly to fill in the empty columns.
</div>

| Scenario | Most appropriate format | Underlying structure | Justification |
|:---------|:------------------------|:---------------------|:--------------|
| An NHS trust exports patient discharge records for analysis in Excel | | | |
| A Barclays mobile banking app receives account details and nested transaction history from a server | | | |
| A Tesco inventory system stores product records with SKU, name, price, and stock count in a relational database | | | |
| A Cancer Research UK developer writes setup instructions for a new data pipeline tool | | | |
| A Transport for London API returns real-time bus arrival predictions to a digital display | | | |
| A university administration system exports student enrolment records to share with a partner institution | | | |

<details>
    <summary style="font-weight:bold">Click here to reveal the solution</summary>

| Scenario | Most appropriate format | Underlying structure | Justification |
|:---------|:------------------------|:---------------------|:--------------|
| An NHS trust exports patient discharge records for analysis in Excel | CSV | Table | Flat tabular data with consistent columns; CSV opens directly in Excel without conversion |
| A Barclays mobile banking app receives account details and nested transaction history from a server | JSON | Tree (nested key-value pairs) | The record is hierarchical — account details at the top level, transaction objects nested inside; JSON represents this naturally and is the standard for API responses |
| A Tesco inventory system stores product records with SKU, name, price, and stock count in a relational database | SQL Table | Table | The data is relational and will be queried frequently; a SQL table provides structured storage, type enforcement, and efficient querying |
| A Cancer Research UK developer writes setup instructions for a new data pipeline tool | Markdown | — | This is documentation for human readers, not machine-processable data; Markdown is the standard format for README files and technical instructions in code repositories |
| A Transport for London API returns real-time bus arrival predictions to a digital display | JSON | Key-Value Pair / Tree | API responses use JSON by convention; bus arrival data may include nested fields (route, stop, predictions array); JSON handles this and is native to web services |
| A university administration system exports student enrolment records to share with a partner institution | CSV | Table | Flat records with consistent columns; CSV is universally readable, does not depend on compatible database systems, and is the standard for inter-institutional data sharing |

</details>

<div class="alert alert-block alert-warning">
<b>Module Challenge connection:</b><br>
Classifying data formats and explaining how your organisation stores, exchanges, and transforms data is key evidence for Module Challenge Part A. Your completed tables and reflections from this notebook — alongside those from Section 1 & 2 — demonstrate the ability to identify formats, select appropriate ones for different use cases, and connect them to underlying data structures.
</div>

---

## Reflection

Reflect on the following question and record your thoughts in the cell below. Ground your answer in an example from your workplace, using **approximately 150 words** in total.

Your answer here forms part of your Module Challenge Part A portfolio and is assessed against **MLO2:** *Critically reflect upon data management systems and their role in digital and technology solutions*.

Where you draw on external sources, cite them in Harvard style. This does not count towards your word count.

Once you commit your notebook, your Coach will be able to provide feedback on your response. You can develop your answer further before your final submission.

<div class="alert alert-block alert-info">

1. Describe a situation in your workplace where data moves between formats. For example, from a database to a spreadsheet, or from an API to a report. Why are these conversions necessary? What challenges might arise during this conversion? Think about what information can be lost or changed when moving between, for example, a nested format (JSON) and a flat one (CSV).

</div>

*Write your responses here.*

1. 

References (if applicable):

---

## Summary

**You have completed Week 2 Section 3.** You can now:

- Viewed the raw CSV text behind the weather dataset and an NHS clinic appointments example, then loaded the same data as a pandas DataFrame and a SQL table
- Converted a weather observation and a nested NHS patient referral record to JSON using `json.dumps()`, comparing JSON's nested structure with CSV's flat structure
- Identified the Markdown syntax used throughout this notebook's own text cells, then used a CSV/JSON/Markdown/SQL comparison table to select the right format for different scenarios
- Traced one dataset through four representations (CSV text, JSON, a DataFrame, and a SQL table) and completed the full mapping from structure to Python to file format to SQL

**Key takeaways:**

- CSV separates columns with commas and rows with newlines; it is flat and cannot represent nested records
- JSON represents the same key-value structure as a Python dictionary, and can nest objects and arrays to represent hierarchical records that CSV cannot
- Format is independent of meaning: the same dataset can move between CSV, JSON, a DataFrame, and a SQL table without the information changing, only the representation

**What comes next:**

- **Section 4** introduces SQL. You will write your own `CREATE TABLE`, `INSERT`, and `SELECT` queries, building on the format and structure knowledge from Sections 2 and 3. The SQL you saw in preview cells throughout this notebook is the same SQL you will write yourself.
- **Workshop 1** (Jupyter and SQL) provides hands-on practice with the Pagila database.

**Action items:**

1. Save this notebook (Ctrl+S or Cmd+S)
2. Commit and push your work to your GitHub repository with the commit message `"Complete Week 2 Section 3 data formats exploration"`
3. Make sure the repository shows your latest commit and that all files are up to date
4. Share a link to your repository with your Coach in the DT401.2 component on Aptem
5. Make sure your Coach has access to your repository so they can view your work and provide feedback
6. Open `DT401_Week_2_Section_4.ipynb` when ready to continue
